# Building a Simple Phase Diagram

This example shows how the `gauge_fix` and `legendre_transform` functions combine to form a simple T-x phase diagram. 
We construct a minimal binary system with two ideal phases (Liquid and Solid) where the pure component reference states are temperature dependent.


In [ ]:
import torch
from zgraph import *
import plotly.graph_objects as go

In [ ]:
# Define coordinates: T, mu1, mu2
T, mu1, mu2 = SignalNodes(0, 1, 2)
R = 8.314
RT = FactorNode([[R]], [T])

# Ideal mixing phases with temperature-dependent references
# We set Liquid as the high-temperature reference: G_L = 0 for both components.
# Solid is stable at low temperatures.
# Component 1: Tm = 1000 K, dS = -10 J/mol.K -> dH = -10000 J/mol
# Component 2: Tm = 1500 K, dS = -10 J/mol.K -> dH = -15000 J/mol
# Free energy of melting: G_S - G_L = dH - T*dS

# Reference energy nodes for the Solid phase (G_L is implicitly 0)
def g_s_1(x):
    return -10000.0 + 10.0 * x[0]

def g_s_2(x):
    return -15000.0 + 10.0 * x[0]

G1_S = DynamicLeafNode(g_s_1, [0])
G2_S = DynamicLeafNode(g_s_2, [0])

# Effective chemical potentials (w = mu - G_ref)
# Liquid (G_ref = 0):
w1_L = FactorNode([[1.0]], [mu1])
w2_L = FactorNode([[1.0]], [mu2])

# Solid: w = mu - G_S
w1_S = FactorNode([[1.0, -1.0]], [mu1, G1_S])
w2_S = FactorNode([[1.0, -1.0]], [mu2, G2_S])

# Phases: logsumexp over components (ideal mixing)
phase_L = FactorNode(torch.eye(2), [w1_L, w2_L], beta=RT)
phase_S = FactorNode(torch.eye(2), [w1_S, w2_S], beta=RT)

# Total System: equilibrium is the minimum of the two phases (SoftMin with beta->0)
system = FactorNode(torch.eye(2), [phase_L, phase_S], beta=0.0)

fcns = [phase_L, phase_S, system]


## Find the Equilibrium Manifold (Gauge Fix)
Since ZGraph natively operates in implicit state variables (chemical potentials), our first step is to constrain the system to the equilibrium manifold.
We create a mesh over Temperature and chemical potential difference, and use `gauge_fix` to shift the coordinates such that the total system grand potential is exactly 0.


In [ ]:
# Compile the base graphs for execution
fcns_compiled = graph_to_function(fcns, compile=True)

# Create inputs
T_vals = torch.linspace(500, 2000, 150)
mu_diff = torch.linspace(-30000, 30000, 150)

T_grid, mu_grid = torch.meshgrid(T_vals, mu_diff, indexing='ij')

# Since it's a binary system, we parameterize by the difference in chemical potentials.
# We set mu1 = mu_grid/2, mu2 = -mu_grid/2
inputs = torch.stack([T_grid, mu_grid/2, -mu_grid/2], dim=-1)

# Flatten inputs for the batched compiled function
inputs_flat = inputs.view(-1, 3)

# Gauge Fix: project onto equilibrium manifold
shifted_inputs_flat = gauge_fix(fcns_compiled[2], inputs_flat, [1, 2])


## Legendre Transform
With our coordinates safely anchored to the equilibrium manifold, we can now apply the Legendre transform to map the chemical potentials `[mu1, mu2]` (indices 1, 2) to their conjugate mole fractions `[x1, x2]`.


In [ ]:
lt_fcns = legendre_transform(fcns, [1, 2])
lt_fcns_compiled = graph_to_function(lt_fcns, compile=True)

# Evaluate Legendre transforms on the gauge-fixed inputs
# Returns (free_energy, dual_coords) where dual_coords are [T, x1, x2]
system_lt_vals_flat = lt_fcns_compiled[2](shifted_inputs_flat)
free_energy_flat, dual_coords_flat = to_numpy(system_lt_vals_flat)

free_energy = free_energy_flat.reshape(150, 150)
dual_coords = dual_coords_flat.reshape(150, 150, 3)


## T-x Phase Diagram
By sweeping the chemical potential difference at various temperatures and mapping it to the conjugate mole fractions via the Legendre transform, the phase boundaries naturally emerge as gaps in the composition space.


In [ ]:
T_np = to_numpy(T_grid)
x1_np = dual_coords[..., 1] # x1 is at index 1

fig = go.Figure()

# We plot the dual variable x1 (mole fraction) against Temperature
fig.add_trace(go.Scatter(
    x=x1_np.flatten(),
    y=T_np.flatten(),
    mode='markers',
    marker=dict(
        size=2,
        color=T_np.flatten(),
        colorscale='Thermal',
        showscale=False
    ),
    name='Equilibrium States'
))

fig.update_layout(
    title='Binary T-x Phase Diagram (Ideal Mixing)',
    xaxis_title='Mole Fraction Component 1 (x1)',
    yaxis_title='Temperature (K)',
    xaxis_range=[-0.05, 1.05],
    yaxis_range=[500, 2000],
    width=700,
    height=600,
    plot_bgcolor='white',
)
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', zeroline=False)
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='LightGray', zeroline=False)

fig.show()
